Notebook to examine the API responses for get_season_games() function vs. get_games_for_date() function so I can apply the same filter to keep only regular season games.

In [2]:
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

import pandas as pd
import requests

In [3]:
def adjust_df_display(dimension, action):
    """This function when called adjusts the output display of pandas dataframes. It either changes the max_columns or max_rows to infinite or resets those
    values to their default display limits.

    Args:
        dimension (string): Display dimension of a dataframe to alter; should be either "columns" or "rows".
        action (string): Action to be carried out on display settings; should be either "max" or "limit".
    """
    if dimension == "columns" and action == "max":
        pd.set_option('display.max_columns', None)
    elif dimension == "rows" and action == "max":
        pd.set_option('display.max_rows', None)
    elif dimension == "columns" and action == "limit":
        pd.reset_option('max_columns')
    else:
        pd.reset_option('max_rows')

In [3]:
adjust_df_display("columns", "max")
adjust_df_display("rows", "max")

In [4]:
NHL_API_BASE_URL = "https://api-web.nhle.com"
API_TIMEOUT_SECONDS = 10
BACKFILL_SEASONS = [
    "20192020",
    "20202021",
    "20212022",
    "20222023",
    "20232024",
    "20242025",
    # "20252026"
]

# Helpers
def _get_team_abbreviations() -> list[str]:
    """
    Fetch all NHL team abbreviations from the standings endpoint.
    Used internally to drive season schedule fetches.
    """
    url = f"{NHL_API_BASE_URL}/v1/standings/now"
    response = requests.get(url, timeout=API_TIMEOUT_SECONDS)
    response.raise_for_status()
    data = response.json()

    abbrevs = []
    for team_record in data.get("standings", []):
        abbrev = team_record.get("teamAbbrev", {}).get("default")
        if abbrev:
            abbrevs.append(abbrev)

    return sorted(set(abbrevs))


def _parse_game_data(game: dict, game_date: str | None = None) -> dict:
    """
    Parse a single game's data from the schedule API response
    into a dictionary ready for DB upsert.

    Parameters:
    - game: raw game dict from schedule API response
    - game_date: optional date string in format 'YYYY-MM-DD'. If not provided,
      falls back to the 'gameDate' field on the game object (club schedule endpoint).
      Pass explicitly when using the schedule-by-date endpoint which does not
      include gameDate on the game object.

    Returns:
    - Dictionary of game fields matching the games table schema
    """

    now_utc = datetime.now(timezone.utc)
    now_et = now_utc.astimezone(ZoneInfo("America/New_York"))

    date = game_date or game.get("gameDate")

    return {
        "game_id": game["id"],
        "season": game["season"],
        "game_type": game["gameType"],
        "game_date": date,
        "game_state": game["gameState"],
        "game_schedule_state": game["gameScheduleState"],
        # Away team
        "away_team_id": game["awayTeam"]["id"],
        "away_team_city": game["awayTeam"]["placeName"]["default"],
        "away_team_name": game["awayTeam"]["commonName"]["default"],
        "away_team_abbrev": game["awayTeam"]["abbrev"],
        "away_team_score": game["awayTeam"].get("score"),
        # Home team
        "home_team_id": game["homeTeam"]["id"],
        "home_team_city": game["homeTeam"]["placeName"]["default"],
        "home_team_name": game["homeTeam"]["commonName"]["default"],
        "home_team_abbrev": game["homeTeam"]["abbrev"],
        "home_team_score": game["homeTeam"].get("score"),
        # Outcome
        "last_period_type": game.get("gameOutcome", {}).get("lastPeriodType"),
        # Timestamps
        "created_at_utc": now_utc,
        "created_at_et": now_et,
    }

In [ ]:
# get_season_games function
games = []
team = 'OTT'
season = '20252026'
url = f"{NHL_API_BASE_URL}/v1/club-schedule-season/{team}/{season}"

response = requests.get(url, timeout=API_TIMEOUT_SECONDS)
response.raise_for_status()
data = response.json()

for game in data.get("games", []):
    if game.get("gameType") != 2:
        continue
    game_id = game["id"]
    
    try:
        games.append(_parse_game_data(game))
    except (KeyError, TypeError) as e:
        print(f"Failed to parse game {game_id} for team {team}: {e}")

# Show structure of one game
games[1]

{'game_id': 2025020027,
 'season': 20252026,
 'game_type': 2,
 'game_date': '2025-10-11',
 'game_state': 'OFF',
 'game_schedule_state': 'OK',
 'away_team_id': 9,
 'away_team_city': 'Ottawa',
 'away_team_name': 'Senators',
 'away_team_abbrev': 'OTT',
 'away_team_score': 2,
 'home_team_id': 13,
 'home_team_city': 'Florida',
 'home_team_name': 'Panthers',
 'home_team_abbrev': 'FLA',
 'home_team_score': 6,
 'last_period_type': 'REG',
 'created_at_utc': datetime.datetime(2026, 7, 28, 15, 53, 0, 669138, tzinfo=datetime.timezone.utc),
 'created_at_et': datetime.datetime(2026, 7, 28, 11, 53, 0, 669138, tzinfo=zoneinfo.ZoneInfo(key='America/New_York'))}

In [ ]:
# get_games_for_date() function
date = '2025-05-10'
url = f"{NHL_API_BASE_URL}/v1/schedule/{date}"
response = requests.get(url, timeout=API_TIMEOUT_SECONDS)
response.raise_for_status()
data = response.json()

games = []
for day in data.get("gameWeek", []):
    if day.get("date") == date:
        for game in day.get("games", []):
            if game.get("gameType") != 2:
                continue
            try:
                games.append(_parse_game_data(game, date))
            except (KeyError, TypeError) as e:
                print(
                    f"Failed to parse game {game.get('id')} for date {date}: {e}"
                )

games

[]

In [5]:
date = '2025-05-10'
url = f"{NHL_API_BASE_URL}/v1/schedule/{date}"
response = requests.get(url, timeout=API_TIMEOUT_SECONDS)
response.raise_for_status()
data = response.json()
data

{'nextStartDate': '2025-05-17',
 'previousStartDate': '2025-05-03',
 'gameWeek': [{'date': '2025-05-10',
   'dayAbbrev': 'SAT',
   'numberOfGames': 2,
   'datePromo': [],
   'games': [{'id': 2024030223,
     'season': 20242025,
     'gameType': 3,
     'venue': {'default': 'Lenovo Center'},
     'neutralSite': False,
     'startTimeUTC': '2025-05-10T22:00:00Z',
     'easternUTCOffset': '-04:00',
     'venueUTCOffset': '-04:00',
     'venueTimezone': 'US/Eastern',
     'gameState': 'OFF',
     'gameScheduleState': 'OK',
     'tvBroadcasts': [{'id': 385,
       'market': 'N',
       'countryCode': 'US',
       'network': 'TNT',
       'sequenceNumber': 11},
      {'id': 501,
       'market': 'N',
       'countryCode': 'US',
       'network': 'truTV',
       'sequenceNumber': 14},
      {'id': 519,
       'market': 'N',
       'countryCode': 'US',
       'network': 'MAX',
       'sequenceNumber': 18},
      {'id': 282,
       'market': 'N',
       'countryCode': 'CA',
       'network': 'S